# FX Multi-ECN Order Book Pipeline (Arrival-Time Aligned)

This notebook builds a reproducible pipeline for comparing ECNs on an arrival-time axis.

In [ ]:

import os, json, numpy as np, pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import grangercausalitytests

USE_SAMPLE = True
DATA_PATH = "/mnt/data/sample_ecn_data.csv"
OUT_DIR = "/mnt/data/ecn_pipeline_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

GRID_MS = 100
DELTA_MS = 100


In [ ]:

df = pd.read_csv(DATA_PATH, parse_dates=["send_time","recv_time"]).sort_values("recv_time").reset_index(drop=True)
venues = sorted(df["venue"].unique())
print("Rows:", len(df), "Venues:", venues)
df.head()


In [ ]:

def compute_latency_stats(df):
    dd = df.copy()
    dd["latency_s"] = (dd["recv_time"] - dd["send_time"]).dt.total_seconds()
    return dd.groupby("venue")["latency_s"].agg(["count","mean","std","median","min","max"]).reset_index()

lat_stats = compute_latency_stats(df)
lat_stats.to_csv(os.path.join(OUT_DIR,"latency_stats.csv"), index=False)
lat_stats.head()


In [ ]:

def build_calendar_grid_merge_asof(df, grid_ms=100):
    dd = df.copy()
    start = dd["recv_time"].min().floor("S")
    end = dd["recv_time"].max().ceil("S")
    grid = pd.date_range(start=start, end=end, freq=f"{grid_ms}ms")
    grid_df = pd.DataFrame({"grid_time": grid})
    venues = sorted(dd["venue"].unique())
    merged_parts = [grid_df]
    for v in venues:
        sub = dd[dd["venue"]==v].sort_values("recv_time")[["recv_time","mid_price","depth"]]
        sub = sub.rename(columns={"mid_price": f"mid_{v}", "depth": f"depth_{v}"})
        tmp = pd.merge_asof(grid_df, sub, left_on="grid_time", right_on="recv_time", direction="backward")
        tmp = tmp.drop(columns=["recv_time"])
        merged_parts.append(tmp[[f"mid_{v}", f"depth_{v}"]])
    combined = pd.concat(merged_parts, axis=1)
    mid_cols = [c for c in combined.columns if c.startswith("mid_")]
    depth_cols = [c for c in combined.columns if c.startswith("depth_")]
    combined["agg_mid"] = (combined[mid_cols].fillna(method="ffill") * combined[depth_cols].fillna(method="ffill")).sum(axis=1) / (combined[depth_cols].fillna(method="ffill").sum(axis=1) + 1e-12)
    return combined

grid_df = build_calendar_grid_merge_asof(df, grid_ms=GRID_MS)
grid_df.to_csv(os.path.join(OUT_DIR, f"aligned_grid_{GRID_MS}ms.csv"), index=False)
grid_df.head()


In [ ]:

def build_features_from_grid(grid_df, venues):
    dd = grid_df.copy()
    for v in venues:
        mid_col = f"mid_{v}"; depth_col = f"depth_{v}"
        if mid_col not in dd.columns: continue
        dd[mid_col] = dd[mid_col].astype(float)
        dd[depth_col] = dd[depth_col].astype(float)
        dd[f"{mid_col}_ret_1"] = dd[mid_col].pct_change(1)
        dd[f"{mid_col}_ret_3"] = dd[mid_col].pct_change(3)
        dd[f"{mid_col}_volatility_10"] = dd[mid_col].rolling(10, min_periods=1).std()
        dd[f"{mid_col}_diff_to_agg"] = dd[mid_col] - dd["agg_mid"]
    depth_cols = [f"depth_{v}" for v in venues if f"depth_{v}" in dd.columns]
    dd["total_depth"] = dd[depth_cols].sum(axis=1)
    for v in venues:
        if f"depth_{v}" in dd.columns:
            dd[f"depth_imbalance_{v}"] = (dd[f"depth_{v}"] - (dd["total_depth"] - dd[f"depth_{v}"])) / (dd["total_depth"] + 1e-9)
    return dd

feat_df = build_features_from_grid(grid_df, venues)
feat_df.to_csv(os.path.join(OUT_DIR, "features_grid.csv"), index=False)
feat_df.head()


In [ ]:

def build_classification_target(df, delta_ms=100, thr=0.0):
    dd = df.copy().sort_values("grid_time").reset_index(drop=True)
    steps = max(1, int(delta_ms/GRID_MS))
    future = dd["agg_mid"].shift(-steps)
    dd["target_dir"] = (future > dd["agg_mid"] + thr).astype(int)
    return dd

def build_regression_target(df, delta_ms=100):
    dd = df.copy().sort_values("grid_time").reset_index(drop=True)
    steps = max(1, int(delta_ms/GRID_MS))
    dd["future_agg"] = dd["agg_mid"].shift(-steps)
    dd["target_delta"] = dd["future_agg"] - dd["agg_mid"]
    return dd

labeled_cls = build_classification_target(feat_df, delta_ms=DELTA_MS, thr=0.0)
labeled_reg = build_regression_target(feat_df, delta_ms=DELTA_MS)
labeled_cls.to_csv(os.path.join(OUT_DIR, f"labeled_cls_{DELTA_MS}ms.csv"), index=False)
labeled_reg.to_csv(os.path.join(OUT_DIR, f"labeled_reg_{DELTA_MS}ms.csv"), index=False)
print("Class balance:", labeled_cls["target_dir"].value_counts(dropna=False).to_dict())
labeled_cls[["grid_time","agg_mid","target_dir"]].head()


In [ ]:

def prepare_cls_data(df, venue):
    mid_col = f"mid_{venue}"; depth_col = f"depth_{venue}"
    feats = [c for c in [mid_col, f"{mid_col}_ret_1", f"{mid_col}_ret_3", f"{mid_col}_volatility_10", f"{mid_col}_diff_to_agg", depth_col, f"depth_imbalance_{venue}"] if c in df.columns]
    dd = df.dropna(subset=feats + ["target_dir"]).copy()
    if dd.empty: return None, None, feats
    X = dd[feats].values; y = dd["target_dir"].values
    return X, y, feats

def eval_cls_venue(df, venue):
    out = {"venue": venue, "auc_mean": None, "f1_mean": None, "acc_mean": None, "valid_splits": 0}
    pack = prepare_cls_data(df, venue)
    if pack[0] is None: return out
    X, y, feats = pack
    tscv = TimeSeriesSplit(n_splits=3)
    aucs, f1s, accs = [], [], []
    scaler = StandardScaler()
    for tr, te in tscv.split(X):
        ytr, yte = y[tr], y[te]
        if len(np.unique(ytr))<2 or len(np.unique(yte))<2: continue
        Xtr, Xte = scaler.fit_transform(X[tr]), scaler.transform(X[te])
        clf = LogisticRegression(max_iter=500).fit(Xtr, ytr)
        p = clf.predict_proba(Xte)[:,1]; yhat = clf.predict(Xte)
        aucs.append(roc_auc_score(yte, p)); f1s.append(f1_score(yte, yhat, zero_division=0)); accs.append(accuracy_score(yte, yhat))
    if len(aucs):
        out.update({"auc_mean": float(np.mean(aucs)), "f1_mean": float(np.mean(f1s)), "acc_mean": float(np.mean(accs)), "valid_splits": len(aucs)})
    return out

cls_results = [eval_cls_venue(labeled_cls, v) for v in venues]
cls_df = pd.DataFrame(cls_results)
cls_df.to_csv(os.path.join(OUT_DIR, "per_ecn_cls_results.csv"), index=False)
cls_df


In [ ]:

def prepare_reg_data(df, venue):
    mid_col = f"mid_{venue}"; depth_col = f"depth_{venue}"
    feats = [c for c in [mid_col, f"{mid_col}_ret_1", f"{mid_col}_ret_3", f"{mid_col}_volatility_10", f"{mid_col}_diff_to_agg", depth_col, f"depth_imbalance_{venue}"] if c in df.columns]
    dd = df.dropna(subset=feats + ["target_delta"]).copy()
    if dd.empty: return None, None, feats
    X = dd[feats].values; y = dd["target_delta"].values
    return X, y, feats

def eval_reg_venue(df, venue):
    out = {"venue": venue, "mse": None, "mae": None, "r2": None, "valid_splits": 0}
    pack = prepare_reg_data(df, venue)
    if pack[0] is None: return out
    X, y, feats = pack
    tscv = TimeSeriesSplit(n_splits=3)
    mses, maes, r2s = [], [], []
    scaler = StandardScaler()
    for tr, te in tscv.split(X):
        Xtr, Xte = scaler.fit_transform(X[tr]), scaler.transform(X[te])
        ytr, yte = y[tr], y[te]
        if np.isclose(np.std(yte), 0): continue
        reg = Ridge(alpha=1.0).fit(Xtr, ytr)
        pred = reg.predict(Xte)
        mses.append(mean_squared_error(yte, pred)); maes.append(mean_absolute_error(yte, pred)); r2s.append(r2_score(yte, pred))
    if len(mses):
        out.update({"mse": float(np.mean(mses)), "mae": float(np.mean(maes)), "r2": float(np.mean(r2s)), "valid_splits": len(mses)})
    return out

reg_results = [eval_reg_venue(labeled_reg, v) for v in venues]
reg_df = pd.DataFrame(reg_results)
reg_df.to_csv(os.path.join(OUT_DIR, "per_ecn_reg_results.csv"), index=False)
reg_df


In [ ]:

def eval_latency_buckets(raw_df, labeled_df, venue):
    rd = raw_df.copy()
    rd["recv_time"] = pd.to_datetime(rd["recv_time"])
    lat_map = rd[rd["venue"]==venue][["recv_time","send_time"]].sort_values("recv_time")
    dd = labeled_df.copy()
    dd["grid_time"] = pd.to_datetime(dd["grid_time"])
    merged = pd.merge_asof(dd.sort_values("grid_time"), lat_map, left_on="grid_time", right_on="recv_time", direction="backward")
    merged["latency_s"] = (merged["recv_time"] - merged["send_time"]).dt.total_seconds()
    q = merged["latency_s"].quantile([0.25,0.5,0.75]).values
    buckets = {
        "low": merged[merged["latency_s"]<=q[0]],
        "mid": merged[(merged["latency_s"]>q[0]) & (merged["latency_s"]<=q[1])],
        "high": merged[(merged["latency_s"]>q[1]) & (merged["latency_s"]<=q[2])],
        "tail": merged[merged["latency_s"]>q[2]]
    }
    def quick_auc(bdf):
        pack = prepare_cls_data(bdf, venue)
        if pack[0] is None: return None, len(bdf)
        X, y, feats = pack
        if len(np.unique(y))<2: return None, len(bdf)
        Xs = StandardScaler().fit_transform(X)
        clf = LogisticRegression(max_iter=500).fit(Xs, y)
        p = clf.predict_proba(Xs)[:,1]
        return float(roc_auc_score(y, p)), len(bdf)
    res = {b: {"auc": quick_auc(bdf)[0], "count": quick_auc(bdf)[1]} for b,bdf in buckets.items()}
    return res

lat_bucket_summary = {v: eval_latency_buckets(df, labeled_cls, v) for v in venues}
with open(os.path.join(OUT_DIR, "latency_bucket_results.json"), "w") as f:
    json.dump(lat_bucket_summary, f, indent=2)
lat_bucket_summary


In [ ]:

def hasbrouck_info_share(grid_df, nlags=3):
    from statsmodels.tsa.api import VAR
    mid_cols = [c for c in grid_df.columns if c.startswith("mid_")]
    ts = grid_df[mid_cols].dropna().astype(float)
    if ts.shape[0] < (nlags+5):
        return pd.DataFrame({"venue":[c.replace("mid_","") for c in mid_cols], "info_share":[np.nan]*len(mid_cols)})
    ts_log = np.log(ts)
    model = VAR(ts_log)
    try:
        res = model.fit(nlags)
        fevd = res.fevd(5)
        decomp_h = fevd.decomp[0]
        contrib = decomp_h.sum(axis=0)
        info_share = contrib / contrib.sum()
        info_df = pd.DataFrame({"venue":[c.replace("mid_","") for c in mid_cols], "info_share": info_share})
    except Exception as e:
        info_df = pd.DataFrame({"venue":[c.replace("mid_","") for c in mid_cols], "info_share":[np.nan]*len(mid_cols)})
    return info_df

info_df = hasbrouck_info_share(grid_df, nlags=3)
info_df.to_csv(os.path.join(OUT_DIR, "hasbrouck_info_share.csv"), index=False)
info_df


In [ ]:

gc_results = {}
for v in venues:
    pair = grid_df[[f"mid_{v}", "agg_mid"]].dropna().astype(float)
    if pair.shape[0] < 20:
        gc_results[v] = {"note":"too few samples"}
        continue
    try:
        test = grangercausalitytests(pair[["agg_mid", f"mid_{v}"]], maxlag=5, verbose=False)
        pvals = {lag: float(test[lag][0]["ssr_ftest"][1]) for lag in test}
        gc_results[v] = pvals
    except Exception as e:
        gc_results[v] = {"error": str(e)}
with open(os.path.join(OUT_DIR, "granger_results.json"), "w") as f:
    json.dump(gc_results, f, indent=2)
gc_results
